# LiDAR Test
I believe that it may be possible to convert LiDAR data directly into a format accessible by base Sionna, without the need for external repositories like NimbusRT

Hopefully my GPU doesn't crash...

In [ ]:
from enum import Enum
import numpy as np
from typing import Optional, Final, Tuple, Dict, List
import sionna.rt
from pyproj import Transformer
from pyproj.enums import TransformDirection

ORIGIN_LAT_LON: Final[Dict[str, float]] = {"lat": 35.72750947, "lon": -78.69595819, "alt": 0}
SIONNA_OFFSET: Final[List[float]] = [2020.5, 1971.5, 46]

class CoordinateConverter:
    """WGS84 converter between geodetic (lat/lon/alt) and local ENU coordinates."""

    def __init__(self, reference_origin: Optional[Dict[str, float]] = None):
        if not reference_origin:
            reference_origin = ORIGIN_LAT_LON
        self.origin = reference_origin

        pipeline = (
            f"+proj=pipeline "
            f"+step +proj=unitconvert +xy_in=deg +z_in=m +xy_out=rad +z_out=m "
            f"+step +proj=cart +ellps=WGS84 "
            f"+step +proj=topocentric +ellps=WGS84 "                            
            f"+lon_0={self.origin['lon']} +lat_0={self.origin['lat']} +h_0={self.origin['alt']} "
            f"+step +proj=unitconvert +xy_in=m +z_in=m +xy_out=ft +z_out=ft"    
        )

        self.transformer = Transformer.from_pipeline(pipeline)


    def update_reference_origin(self, origin: Dict[str, float]) -> Dict[str, float]:
        self.origin = origin
        pipeline = (
            f"+proj=pipeline "
            f"+step +proj=unitconvert +xy_in=deg +z_in=m +xy_out=rad +z_out=m "
            f"+step +proj=cart +ellps=WGS84 "
            f"+step +proj=topocentric +ellps=WGS84 "                            
            f"+lon_0={self.origin['lon']} +lat_0={self.origin['lat']} +h_0={self.origin['alt']} "
            f"+step +proj=unitconvert +xy_in=m +z_in=m +xy_out=ft +z_out=ft"    
        )

        self.transformer = Transformer.from_pipeline(pipeline)
        return self.origin


    def get_origin(self) -> Dict[str, float]:
        return self.origin


    def lat_lon_alt_to_local(
        self, lat: float, lon: float, alt: float
    ) -> Tuple[float, float, float]:
        """Convert geodetic coordinate to local ENU tuple (x=east, y=north, z=up)."""
        east, north, up = self.transformer.transform(lon, lat, alt, direction=TransformDirection.FORWARD)
        return (east + SIONNA_OFFSET[0], north + SIONNA_OFFSET[1], up + SIONNA_OFFSET[2])


    def local_to_lat_lon_alt(
        self, x: float, y: float, z: float
    ) -> Tuple[float, float, float]:
        lon, lat, alt = self.transformer.transform(x - SIONNA_OFFSET[0],
                                                   y - SIONNA_OFFSET[1],
                                                   z - SIONNA_OFFSET[2], 
                                                   direction=TransformDirection.INVERSE)
        return (lat, lon, alt)

In [ ]:
# Defining the scene and the particular lat/lon positions

cc = CoordinateConverter()

lw1_lla = [35.72750947, -78.69595819, 0]
north_point = [35.728694, -78.695839, 0]
lw1_xyz = cc.lat_lon_alt_to_local(lw1_lla[0], lw1_lla[1], lw1_lla[2])
north_xyz = cc.lat_lon_alt_to_local(north_point[0], north_point[1], north_point[2])
print(lw1_xyz, north_xyz)

env = sionna.rt.load_scene("/home/everetttucker471/Documents/AERPAW-DT-SIONNA-EXTENSION/data/scenes/lake-wheeler-scene.xml")
env.add(sionna.rt.Transmitter(name="tx0", position=lw1_xyz))
env.add(sionna.rt.Transmitter(name="tx1", position=north_xyz))
env.preview(show_devices=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from AERPAWEnvironment import AERPAWEnv

# Defining the scene, including receivers and transmitters

# Defining a generic UAV for simplicity
generic_uav = {
    "device_type": "tx",
    "mass": 5,
    "efficiency": 0.7,
    "position": np.zeros(3),
    "velocity": np.zeros(3),
    "color": np.array([1, 0, 0]),
    "bandwidth": 50,
    "rotor_area": 0.25,
    "signal_power": 3,
    "throughput_capacity": 625000000,
    "battery_capacity": 10000
}

# Configuring some generic uavs with random x and y positions
num_uavs = 2
uavs = []
for i in range(num_uavs):
    uav = dict(generic_uav)
    uav["position"] = np.array([2118, 1875, 60])
    uavs.append(uav)

# Plotting the positions in 3d
fig = plt.figure()
axis = fig.add_subplot(projection='3d')
axis.scatter([uavs[i]["position"][0] for i in range(num_uavs)],
              [uavs[i]["position"][1] for i in range(num_uavs)],
              [uavs[i]["position"][2] for i in range(num_uavs)],
              color="red")
plt.title("Sample UAV Postions")
plt.show()

In [ ]:
"""
# Defining Generic Ground User

generic_gu = {
    "device_type": "rx",
    "position": np.zeros(3),
    "velocity": np.zeros(3),
    "color": np.array([0, 1, 0]),
    "bandwidth": 50,
    "desired_throughput": 375000
}

# Configuring some sample Ground Users
num_gus = 10
gus = []
for i in range(num_gus):
    gu = dict(generic_gu)
    gu["position"] = np.array([np.random.rand() * 100 - 150, np.random.rand() * 100 + 1750, 100])
    gus.append(gu)

# Plotting UAV and Ground User Positions
fig = plt.figure()
axis = fig.add_subplot(projection='3d')
axis.scatter([uavs[i]["position"][0] for i in range(num_uavs)],
              [uavs[i]["position"][1] for i in range(num_uavs)],
              [uavs[i]["position"][2] for i in range(num_uavs)],
              color="red")
axis.scatter([gus[i]["position"][0] for i in range(num_gus)],
           [gus[i]["position"][1] for i in range(num_gus)],
           [gus[i]["position"][2] for i in range(num_gus)],
           color="blue")
plt.title("Sample UAV and Ground User Positions")
plt.show()
"""

In [ ]:
# Generating Sample Base Stations

# Defining a generic base station, then randomizing positions
generic_base_station = {
    "device_type": "rx",
    "position": np.zeros(3),
    "color": np.array([0, 1, 0]),
    "bandwidth": 50,
    "signal_power": 10,
    "throughput_capacity": 625000000,
    "battery_capacity": 1000000
}

# Adding some base stations with random positions
num_bs = 2
bss = []
for i in range(num_bs):
    bs = dict(generic_base_station)
    bs["position"] = np.array([2021, 1974, 40])
    bss.append(bs)

# Plotting UAV, Ground User, and Base Station Positions
fig = plt.figure()
axis = fig.add_subplot(projection='3d')
axis.scatter([uavs[i]["position"][0] for i in range(num_uavs)],
              [uavs[i]["position"][1] for i in range(num_uavs)],
              [uavs[i]["position"][2] for i in range(num_uavs)],
              color="red")
"""
axis.scatter([gus[i]["position"][0] for i in range(num_gus)],
           [gus[i]["position"][1] for i in range(num_gus)],
           [gus[i]["position"][2] for i in range(num_gus)],
           color="blue")
"""
axis.scatter([bss[i]["position"][0] for i in range(num_bs)],
           [bss[i]["position"][1] for i in range(num_bs)],
           [bss[i]["position"][2] for i in range(num_bs)],
           color="green")
plt.title("Sample UAV, Ground User, and Base Station Positions")
plt.show()

In [ ]:
from sionna.rt import Camera
# test_env = AERPAWEnv(scene_path="../data/aerpaw/LakeWheelerTerrain25/lake_wheeler_road_aerpaw.xml",
#                      uavs={}, ground_users={}, base_stations={}, temperature=300)

# Creating the scene with the sample devices
env = AERPAWEnv(scene_path='/home/everetttucker471/Documents/aerpaw_sionna/data/scenes/lake-wheeler-scene.xml',
                uavs=uavs, ground_users={}, base_stations=bss, temperature=300)

# Attempting to set objects
for obj in env.scene.objects.values():
    obj.radio_material.scattering_coefficient = 0.1
    
env.visualize()

# env = AERPAWEnv(scene_path='../data/aerpaw/LakeWheelerTerrain/lake_wheeler_road_aerpaw.xml',
#                 uavs={}, ground_users={}, base_stations={}, temperature=300)

# cam = Camera(position=(0, 0, 1000), look_at=(0, 0, 0))
# env.scene.render(camera=cam)

In [ ]:
# Computing some sample paths to test the geometry of the environment, does it actually work?
import time
import drjit as dr

dr.flush_malloc_cache()
start = time.time()
snr_dict = env.getSNR(max_depth=3, num_samples=100, sampling_frequency=1.0, )  # Using default arguments, should be alright
end = time.time()
print(f"Time to compute SNR: {end - start}")
print(f"Number of Recievers: {len(snr_dict)}")
print(f"Number of Transmitters: {len(snr_dict["bs0"])}")
print(snr_dict["bs0"])

In [ ]:
import numpy as np

# Visualizing paths, this is the real test to see if rays bounce off the geometry of the environment
num_path_list = []
time_list = []
num_rx_tx_list = []
trials = 10

dr.flush_malloc_cache()
# Attempting to set objects
for obj in env.scene.objects.values():
    obj.radio_material.scattering_coefficient = 0.01
    
general_paths = env.computeGeneralPaths(max_depth=3, num_samples=100000, mode='gpu')
num_paths = np.sum(general_paths.valid)
env.visualize(paths=general_paths)

In [ ]:
radio_map = env.computeRadioMap(max_depth=2, num_samples=1000000, cell_size=(5.0, 5.0))
env.visualize(radio_map=radio_map)

In [ ]:
# Imports
import numpy as np;
import drjit as dr;
import matplotlib.pyplot as plt;
from AERPAWEnvironment import AERPAWEnv;

In [ ]:
# Importing some more realistic transmitters and receivers for this scenario

center = np.array([-200, -450, 300])
uavs = []
num_uavs = 5

for i in range(num_uavs):
    uavs.append({
    "device_type": "tx",
    "mass": 5,
    "efficiency": 0.7,
    "position": center + np.random.rand(3) * 50,
    "velocity": np.zeros(3),
    "color": np.array([1, 0, 0]),
    "bandwidth": 50,
    "rotor_area": 0.25,
    "signal_power": 3,
    "throughput_capacity": 625000000,
    "battery_capacity": 10000
})
    
gus = []
num_gu = 10
for i in range(num_gu):
    gus.append({
    "device_type": "rx",
    "position": center + np.random.rand(3) * 60 - np.array([-200, 200, 20]),
    "velocity": np.zeros(3),
    "color": np.array([0, 1, 0]),
    "bandwidth": 50,
    "desired_throughput": 375000
})
    
# Creating the scene with the sample devices
env = AERPAWEnv(scene_path='../data/LiDAR/lidar-sample.xml',
                uavs=uavs, ground_users=gus, base_stations={}, temperature=300)

env.visualize()

In [ ]:
# Computing some sample paths to test the geometry of the environment, does it actually work?
import time

start = time.time()
snr_dict = env.getSNR(max_depth=2, num_samples=5000000, sampling_frequency=1.0, )  # Using default arguments, should be alright
end = time.time()
print(f"Time to compute SNR: {end - start} seconds")
print(f"Number of Recievers: {len(snr_dict)}")
print(f"Number of Transmitters: {len(snr_dict["gu0"])}")
print(snr_dict["gu0"])

In [ ]:
# Visualizing paths, this is the real test to see if ray bounce off the geometry of the environment
dr.flush_malloc_cache()  # To free memory for repeated runs
general_paths = env.computeGeneralPaths(max_depth=2, num_samples=1000000, mode='gpu')
env.visualize(paths=general_paths)

In [ ]:
from sionna.rt import PathSolver

dr.flush_malloc_cache()  # To free memory for repeated runs
p_solver  = PathSolver()

# Computing paths with scene transmitters and receivers, and specified parameters
paths = p_solver(scene=env.scene,
                 max_depth=2,
                 los=True,
                 specular_reflection=True,
                 diffuse_reflection=False,
                 refraction=True,
                 synthetic_array=False,
                 seed=41)

env.visualize(paths=paths)

# Getting the Channel Impulse response
print(paths.cir())